<a href="https://colab.research.google.com/github/anasmita3/SURGE/blob/main/SURGE_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install datasets

In [2]:
from huggingface_hub import login

login("")

In [3]:
from datasets import load_dataset

ds = load_dataset("Vacaspati/Vacaspati")

print(ds)

print("\nColumns:")
print(ds["train"].column_names)

print("\nSample:")
print(ds["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

vacaspati.zip:   0%|          | 0.00/337M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4667936 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4667936
    })
})

Columns:
['text']

Sample:
{'text': 'সাময়িক পত্রে প্রকাশিত'}


In [5]:
import unicodedata
import re

count = 0

with open("bn_literature.txt", "w", encoding="utf-8") as out:

    for row in ds["train"]:

        text = row["text"]

        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"\s+", " ", text).strip()

        if not text:
            continue

        out.write(text + "\n")
        count += 1

print("Saved documents:", count)

Saved documents: 4604919


In [6]:
vocab = set()

chars = 0
words = 0

with open("bn_literature.txt", encoding="utf-8") as f:

    for line in f:

        chars += len(line)

        toks = line.split()

        words += len(toks)

        vocab.update(toks)

print("Characters :", chars)
print("Words      :", words)
print("Vocabulary :", len(vocab))

print(
    "TTR :",
    len(vocab) / words
)

Characters : 628212667
Words      : 102900420
Vocabulary : 3661914
TTR : 0.03558696844969146


In [7]:
import random

random.seed(42)

TARGET = 250000

with open("bn_literature.txt", encoding="utf-8") as f:
    total_docs = sum(1 for _ in f)

print("Total docs:", total_docs)

selected = set(
    random.sample(
        range(total_docs),
        TARGET
    )
)

count = 0

with open("bn_literature.txt", encoding="utf-8") as inp, \
     open("bn_literature_250k.txt", "w", encoding="utf-8") as out:

    for idx, line in enumerate(inp):

        if idx in selected:

            out.write(line)
            count += 1

print("Saved:", count)

Total docs: 4604919
Saved: 250000


In [8]:
vocab = set()

chars = 0
words = 0

with open("bn_literature_250k.txt", encoding="utf-8") as f:

    for line in f:

        chars += len(line)

        toks = line.split()

        words += len(toks)

        vocab.update(toks)

print("Characters :", chars)
print("Words      :", words)
print("Vocabulary :", len(vocab))

print("TTR :", len(vocab)/words)

print(
    "Average Word Length:",
    chars / words
)

Characters : 34204085
Words      : 5601171
Vocabulary : 522341
TTR : 0.09325567814301688
Average Word Length: 6.106595388714253


In [9]:
with open(
    "bn_literature_250k.txt",
    encoding="utf-8"
) as f:

    literature_docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

print(
    "Literature:",
    len(literature_docs)
)

Literature: 250000


In [10]:
import random

random.seed(42)

literature_sample = random.sample(
    literature_docs,
    62617
)

print(
    "Literature sample:",
    len(literature_sample)
)

Literature sample: 62617


In [11]:
import kagglehub
import os

path = kagglehub.dataset_download(
    "mdsalmanhossain/bangla-social-media-abuse-dataset"
)

print(path)

100%|██████████| 4.44M/4.44M [00:00<00:00, 136MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/mdsalmanhossain/bangla-social-media-abuse-dataset/versions/1


In [12]:
import os

for root, dirs, files in os.walk(path):
    for f in files:
        print(f)

cyberbulling bangla dataset.xlsx


In [13]:
import pandas as pd
import os

xlsx_file = os.path.join(
    path,
    "cyberbulling bangla dataset.xlsx"
)

df = pd.read_excel(xlsx_file)

print(df.columns.tolist())

['comment', 'Category', 'Gender', 'comment react number', 'label']


In [14]:
comments = (
    df["comment"]
    .dropna()
    .astype(str)
    .str.strip()
)

comments = [
    x for x in comments
    if x
]

print(
    "Documents:",
    len(comments)
)

Documents: 44001


In [15]:
with open(
    "social_media_44k.txt",
    "w",
    encoding="utf-8"
) as f:

    for line in comments:
        f.write(line + "\n")

print("Saved")

Saved


In [16]:
import os

print(
    "Documents:",
    sum(
        1 for _ in open(
            "social_media_44k.txt",
            encoding="utf-8"
        )
    )
)

print(
    "Size MB:",
    round(
        os.path.getsize(
            "social_media_44k.txt"
        )/(1024**2),
        2
    )
)

Documents: 44114
Size MB: 10.48


In [19]:
with open(
    "social_media_44k.txt",
    encoding="utf-8"
) as f:

    social_docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

print("Social:", len(social_docs))

Social: 44113


In [20]:
social_sample = []

while len(social_sample) < 62617:
    social_sample.extend(social_docs)

social_sample = social_sample[:62617]

print(len(social_sample))

62617


In [21]:
combined = literature_sample + social_sample

import random
random.shuffle(combined)

print(len(combined))

125234


In [23]:
with open(
    "literature_social_125k.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in combined:
        f.write(doc + "\n")

print("Saved")

Saved


In [24]:
from collections import Counter
import os

with open(
    "literature_social_125k.txt",
    encoding="utf-8"
) as f:

    text = f.read()

chars = len(text)

words = text.split()

vocab = set(words)

print("Characters :", chars)
print("Words      :", len(words))
print("Vocabulary :", len(vocab))
print("TTR        :", len(vocab)/len(words))
print(
    "Average Word Length:",
    sum(len(w) for w in words)/len(words)
)

Characters : 14518686
Words      : 2425965
Vocabulary : 274747
TTR        : 0.11325266440364969
Average Word Length: 4.978056979387584


In [25]:
import random

random.seed(42)

with open(
    "literature_social_125k.txt",
    encoding="utf-8"
) as f:
    docs = [x.strip() for x in f if x.strip()]

random.shuffle(docs)

split = int(0.9 * len(docs))

train_docs = docs[:split]
test_docs = docs[split:]

print("Train:", len(train_docs))
print("Test :", len(test_docs))

Train: 112710
Test : 12524


In [26]:
with open("train_litsocial_bn.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_docs))

with open("test_litsocial_bn.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(test_docs))

In [27]:
import os

print(
    "Train MB:",
    round(os.path.getsize("train_litsocial_bn.txt")/(1024**2),2)
)

print(
    "Test MB:",
    round(os.path.getsize("test_litsocial_bn.txt")/(1024**2),2)
)

Train MB: 32.52
Test MB: 3.61


Transliteration

In [28]:
!pip install aksharamukha -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 35.5 MB/s eta 0:00:00


In [29]:
from aksharamukha import transliterate

In [30]:
def transliterate_file(
    infile,
    outfile
):

    count = 0

    with open(
        infile,
        encoding="utf-8"
    ) as fin, open(
        outfile,
        "w",
        encoding="utf-8"
    ) as fout:

        for line in fin:

            line = line.strip()

            if not line:
                continue

            iso = transliterate.process(
                "Bengali",
                "ISO",
                line
            )

            fout.write(
                iso + "\n"
            )

            count += 1

            if count % 10000 == 0:

                print(
                    f"{count:,} docs"
                )

    print(
        "Completed:",
        count
    )

In [31]:
transliterate_file(
    "train_litsocial_bn.txt",
    "train_litsocial_iso.txt"
)

10,000 docs
20,000 docs
30,000 docs
40,000 docs
50,000 docs
60,000 docs
70,000 docs
80,000 docs
90,000 docs
100,000 docs
110,000 docs
Completed: 112710


In [32]:
transliterate_file(
    "test_litsocial_bn.txt",
    "test_litsocial_iso.txt"
)

10,000 docs
Completed: 12524


In [33]:
from collections import Counter

def corpus_stats(path):

    chars = 0
    words = 0
    vocab = Counter()

    with open(
        path,
        encoding="utf-8"
    ) as f:

        for line in f:

            chars += len(line)

            toks = line.split()

            words += len(toks)

            vocab.update(toks)

    return {
        "chars": chars,
        "words": words,
        "vocab": len(vocab),
        "avg_word_len": chars / words
    }

In [34]:
bn = corpus_stats(
    "train_litsocial_bn.txt"
)

print("Bengali:", bn)

Bengali: {'chars': 13067441, 'words': 2183362, 'vocab': 257501, 'avg_word_len': 5.985008899119798}


In [35]:
iso = corpus_stats(
    "train_litsocial_iso.txt"
)

print("ISO:", iso)

ISO: {'chars': 15183397, 'words': 2183353, 'vocab': 252394, 'avg_word_len': 6.954164993017621}


In [36]:
print(
    "Expansion Factor:",
    iso["chars"] / bn["chars"]
)

print(
    "Vocabulary Ratio:",
    iso["vocab"] / bn["vocab"]
)

Expansion Factor: 1.161925812406576
Vocabulary Ratio: 0.9801670673123599


In [37]:
with open(
    "test_litsocial_bn.txt",
    encoding="utf-8"
) as f:

    bn_lines = [
        x.strip()
        for x in f
        if x.strip()
    ]

with open(
    "test_litsocial_iso.txt",
    encoding="utf-8"
) as f:

    iso_lines = [
        x.strip()
        for x in f
        if x.strip()
    ]

print(
    len(bn_lines),
    len(iso_lines)
)

12524 12524


In [38]:
import random

random.seed(42)

idxs = random.sample(
    range(len(bn_lines)),
    100
)

correct = 0

for idx in idxs:

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso_lines[idx]
    )

    if back == bn_lines[idx]:
        correct += 1

print(
    "Exact Match:",
    correct,
    "/ 100"
)

print(
    "Accuracy:",
    correct/100
)

Exact Match: 51 / 100
Accuracy: 0.51


In [39]:
shown = 0

for idx in idxs:

    original = bn_lines[idx]

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso_lines[idx]
    )

    if original != back:

        print("="*80)

        print("ORIGINAL:")
        print(original)

        print()

        print("BACK:")
        print(back)

        shown += 1

        if shown == 10:
            break

ORIGINAL:
সাহিত্যিক বলিল, আপনারা জানেন না, আর জানিবেনই বা কিরূপে, আপনারা তো সাহিত্যিক নহেন, সাহিত্যিকদের মাথা বড় শক্ত। হেন ছাদ নাই—খসিয়া পড়িয়া যাহা তাহাদের মাথা ফাটাইতে সমর্থ, হেন ভূমিকম্প নাই যাহাতে তাহারা টলে, হেন অগ্নিকাণ্ড নাই—যাহাতে তাহারা পোড়ে

BACK:
সাহিত্যিক বলিল, আপনারা জানেন না, আর জানিবেনই বা কিরূপে, আপনারা তো সাহিত্যিক নহেন, সাহিত্যিকদের মাথা বড় শক্ত। হেন ছাদ নাই—খসিয়া পড়িয়া যাহা তাহাদের মাথা ফাটাইতে সমর্থ, হেন ভূমিকম্প নাই যাহাতে তাহারা টলে, হেন অগ্নিকাণ্ড নাই—যাহাতে তাহারা পোড়ে
ORIGINAL:
গান ভালো লাগছে।। অপেক্ষায় রইলাম আরো ভালো কিছু গানের জন্য

BACK:
গান ভালো লাগছে॥ অপেক্ষায় রইলাম আরো ভালো কিছু গানের জন্য
ORIGINAL:
এই সময় জীবনযাপন ডেস্ক: মেঘ-আর পাহাড়ের গল্প তো সবারই জানা। মেঘের ছোঁয়ায় পাহাড় কী ভাবে হয়ে উঠেছিল ছাব্বিশ বছরের ছোকরা সেই গল্পও অনেকে শুনেছেন। ‘ভালোবাসলে নারীরা হয়ে যায় নরম নদী, পুরুষেরা জ্বলন্ত কাঠ’- পূর্ণেন্দু পত্রীর ‘সেই গল্পটার’ কথা সকলেই জানেন। প্রেমে পড়লে কিংবা ভালোবাসলে বদলে যায় মেয়েরা। আর সত্যিকারের প্রেমিকের খোঁজ পেলে মেয়েরা কিন্তু তাঁক

Tokenization

In [40]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_litsocial_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_bn_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [41]:
import numpy as np
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1
                tokens += len(tks)
                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [42]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_bn_{vocab_size}.json",
        "test_litsocial_bn.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 1.2365881707975582e-05, 'fertility': 1.6963805064240756, 'cpt': 2.933431661511322, 'compression': 2.933431661511322, 'avg_tokens': np.float64(1.6963805064240756), 'wfr': 0.4192775851906201, 'variance': np.float64(1.5464140662839745)}
{'vocab': 10000, 'oov': 1.2365881707975582e-05, 'fertility': 1.5254922651409917, 'cpt': 3.2620396715384676, 'compression': 3.2620396715384676, 'avg_tokens': np.float64(1.5254922651409917), 'wfr': 0.34087789516205486, 'variance': np.float64(1.1835925074555365)}
{'vocab': 15000, 'oov': 1.2365881707975582e-05, 'fertility': 1.4530323202928241, 'cpt': 3.424711355706221, 'compression': 3.424711355706221, 'avg_tokens': np.float64(1.4530323202928241), 'wfr': 0.30531774133048645, 'variance': np.float64(1.0145776300110738)}
{'vocab': 20000, 'oov': 1.2365881707975582e-05, 'fertility': 1.4109924444462765, 'cpt': 3.5267490673685624, 'compression': 3.5267490673685624, 'avg_tokens': np.float64(1.4109924444462765), 'wfr': 0.284460620849701, 'varianc

In [43]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "bpe_bn_50000.json"
)

samples = [
    "বাংলাদেশ",
    "প্রতিষ্ঠান",
    "বিশ্ববিদ্যালয়",
    "মুক্তিযুদ্ধ",
    "আলম ভাই ভালো মানুষ",
    "খানকি মাগী",
    "কম্পিউটার বিজ্ঞান",
    "কৃত্রিম বুদ্ধিমত্তা"
]

for s in samples:

    enc = tok.encode(s)

    print("="*60)
    print("TEXT :", s)
    print("TOKENS:")
    print(enc.tokens)

TEXT : বাংলাদেশ
TOKENS:
['বাংলাদেশ']
TEXT : প্রতিষ্ঠান
TOKENS:
['প্রতিষ্ঠান']
TEXT : বিশ্ববিদ্যালয়
TOKENS:
['বিশ্ববিদ্যালয়']
TEXT : মুক্তিযুদ্ধ
TOKENS:
['মুক্তিযুদ্ধ']
TEXT : আলম ভাই ভালো মানুষ
TOKENS:
['আলম', 'ভাই', 'ভালো', 'মানুষ']
TEXT : খানকি মাগী
TOKENS:
['খানকি', 'মাগী']
TEXT : কম্পিউটার বিজ্ঞান
TOKENS:
['কম্পিউটার', 'বিজ্ঞান']
TEXT : কৃত্রিম বুদ্ধিমত্তা
TOKENS:
['কৃত্রিম', 'বুদ্ধিমত্তা']


In [44]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "bpe_bn_50000.json"
)

print(
    "Vocabulary Size:",
    len(tok.get_vocab())
)

Vocabulary Size: 50000


In [45]:
samples = [
    "বাংলাদেশ",
    "মুক্তিযুদ্ধ",
    "বিশ্ববিদ্যালয়",
    "আলম ভাই ভালো মানুষ"
]

for s in samples:

    enc = tok.encode(s)

    decoded = tok.decode(
        enc.ids
    )

    print("="*50)
    print("Original :", s)
    print("Decoded  :", decoded)

Original : বাংলাদেশ
Decoded  : বাংলাদেশ
Original : মুক্তিযুদ্ধ
Decoded  : মুক্তিযুদ্ধ
Original : বিশ্ববিদ্যালয়
Decoded  : বিশ্ববিদ্যালয়
Original : আলম ভাই ভালো মানুষ
Decoded  : আলম ভাই ভালো মানুষ


In [46]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_litsocial_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_iso_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [47]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "bpe_iso_50000.json"
)

samples = [
    "bāṃlādeśa",
    "muktijuddha",
    "viśvavidyālaya",
    "kr̥trima bud'dhimattā",
    "ālama bhāi bhālō mānuṣa"
]

for s in samples:

    enc = tok.encode(s)

    print("="*60)
    print("TEXT :", s)
    print("TOKENS:")
    print(enc.tokens)

TEXT : bāṃlādeśa
TOKENS:
['bā', 'ṃ', 'lā', 'de', 'śa']
TEXT : muktijuddha
TOKENS:
['mukti', 'ju', 'ddha']
TEXT : viśvavidyālaya
TOKENS:
['viś', 'va', 'vi', 'dyā', 'la', 'ya']
TEXT : kr̥trima bud'dhimattā
TOKENS:
['kr̥trima', 'bud', "'", 'dhi', 'mattā']
TEXT : ālama bhāi bhālō mānuṣa
TOKENS:
['ālama', 'bhāi', 'bhālō', 'mānuṣa']


In [48]:
samples = [
    "bāṃlādeśa",
    "muktijuddha",
    "viśvavidyālaya"
]

for s in samples:

    enc = tok.encode(s)

    decoded = tok.decode(
        enc.ids
    )

    print("="*50)
    print("Original :", s)
    print("Decoded  :", decoded)

Original : bāṃlādeśa
Decoded  : bā ṃ lā de śa
Original : muktijuddha
Decoded  : mukti ju ddha
Original : viśvavidyālaya
Decoded  : viś va vi dyā la ya


In [49]:
import numpy as np
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1
                tokens += len(tks)
                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [50]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_iso_{vocab_size}.json",
        "test_litsocial_iso.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 1.236608559804451e-05, 'fertility': 1.682995395694129, 'cpt': 3.534253587497214, 'compression': 3.534253587497214, 'avg_tokens': np.float64(1.682995395694129), 'wfr': 0.41306847926001344, 'variance': np.float64(1.474778382869909)}
{'vocab': 10000, 'oov': 1.236608559804451e-05, 'fertility': 1.5224011640608577, 'cpt': 3.907073020824026, 'compression': 3.907073020824026, 'avg_tokens': np.float64(1.5224011640608577), 'wfr': 0.33802694982254666, 'variance': np.float64(1.1425191813400355)}
{'vocab': 15000, 'oov': 1.236608559804451e-05, 'fertility': 1.454255788358567, 'cpt': 4.090155640148413, 'compression': 4.090155640148413, 'avg_tokens': np.float64(1.454255788358567), 'wfr': 0.3045560781371729, 'variance': np.float64(1.00157916401679)}
{'vocab': 20000, 'oov': 1.236608559804451e-05, 'fertility': 1.4152531543823346, 'cpt': 4.202875292349543, 'compression': 4.202875292349543, 'avg_tokens': np.float64(1.4152531543823346), 'wfr': 0.2848404156653572, 'variance': np.float64

In [51]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_litsocial_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_bn_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [52]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "wp_bn_50000.json"
)

samples = [
    "বাংলাদেশ",
    "মুক্তিযুদ্ধ",
    "বিশ্ববিদ্যালয়",
    "আলম ভাই ভালো মানুষ",
    "খানকি মাগী"
]

for s in samples:

    enc = tok.encode(s)

    print("="*60)
    print("TEXT :", s)
    print(enc.tokens)

TEXT : বাংলাদেশ
['বাংলাদেশ']
TEXT : মুক্তিযুদ্ধ
['মুক্তিযুদ্ধ']
TEXT : বিশ্ববিদ্যালয়
['বিশ্ববিদ্যালয়']
TEXT : আলম ভাই ভালো মানুষ
['আলম', 'ভাই', 'ভালো', 'মানুষ']
TEXT : খানকি মাগী
['খানকি', 'মাগী']


In [53]:
import numpy as np
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1
                tokens += len(tks)
                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [54]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_bn_{vocab_size}.json",
        "test_litsocial_bn.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 1.6487842277300775e-05, 'fertility': 1.7948665103069623, 'cpt': 2.7724715230571375, 'compression': 2.7724715230571375, 'avg_tokens': np.float64(1.7948665103069623), 'wfr': 0.45160199997526823, 'variance': np.float64(1.8417716464839111)}
{'vocab': 10000, 'oov': 1.6487842277300775e-05, 'fertility': 1.5812747575256696, 'cpt': 3.1469649811533227, 'compression': 3.1469649811533227, 'avg_tokens': np.float64(1.5812747575256696), 'wfr': 0.3597111330033017, 'variance': np.float64(1.376554349981194)}
{'vocab': 15000, 'oov': 1.6487842277300775e-05, 'fertility': 1.4948578541897668, 'cpt': 3.3288892810562047, 'compression': 3.3288892810562047, 'avg_tokens': np.float64(1.4948578541897668), 'wfr': 0.32022687270973565, 'variance': np.float64(1.1737708732913514)}
{'vocab': 20000, 'oov': 1.6487842277300775e-05, 'fertility': 1.4445699352439993, 'cpt': 3.444773538551092, 'compression': 3.444773538551092, 'avg_tokens': np.float64(1.4445699352439993), 'wfr': 0.2961669888665845, 'varia

In [55]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_litsocial_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_iso_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [56]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "wp_iso_50000.json"
)

samples = [
    "bāṃlādeśa",
    "muktijuddha",
    "viśvavidyālaya",
    "ālama bhāi bhālō mānuṣa"
]

for s in samples:

    enc = tok.encode(s)

    print("="*60)
    print("TEXT :", s)
    print(enc.tokens)

TEXT : bāṃlādeśa
['bā', '##ṃ', '##lā', '##de', '##śa']
TEXT : muktijuddha
['mukti', '##ju', '##ddha']
TEXT : viśvavidyālaya
['viś', '##va', '##vid', '##yāla', '##ya']
TEXT : ālama bhāi bhālō mānuṣa
['ālama', 'bhāi', 'bhālō', 'mānuṣa']


In [57]:
import numpy as np
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1
                tokens += len(tks)
                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [58]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_iso_{vocab_size}.json",
        "test_litsocial_iso.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 1.6488114130726014e-05, 'fertility': 1.7795621581292587, 'cpt': 3.342469656258686, 'compression': 3.342469656258686, 'avg_tokens': np.float64(1.7795621581292587), 'wfr': 0.44263578992493785, 'variance': np.float64(1.7722225775557054)}
{'vocab': 10000, 'oov': 1.6488114130726014e-05, 'fertility': 1.5782175524218978, 'cpt': 3.7688926383092087, 'compression': 3.7688926383092087, 'avg_tokens': np.float64(1.5782175524218978), 'wfr': 0.3554672525443221, 'variance': np.float64(1.3176622031995937)}
{'vocab': 15000, 'oov': 1.6488114130726014e-05, 'fertility': 1.4949937963470583, 'cpt': 3.9787004665218206, 'compression': 3.9787004665218206, 'avg_tokens': np.float64(1.4949937963470583), 'wfr': 0.3180969418670316, 'variance': np.float64(1.1242901659349933)}
{'vocab': 20000, 'oov': 1.6488114130726014e-05, 'fertility': 1.4470958247972991, 'cpt': 4.1103929767791625, 'compression': 4.1103929767791625, 'avg_tokens': np.float64(1.4470958247972991), 'wfr': 0.2955288356506004, 'varia

In [74]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000, 10000, 15000, 20000, 25000,
    30000, 35000, 40000, 45000, 50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(Unigram())

    tokenizer.pre_tokenizer = Whitespace()

    trainer = UnigramTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"],
        unk_token="[UNK]"
    )

    tokenizer.train(
        ["train_litsocial_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"uni_bn_{vocab_size}.json"
    )

    print("Saved:", vocab_size)

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [75]:
from tokenizers import Tokenizer

for v in VOCABS:

    tok = Tokenizer.from_file(
        f"uni_bn_{v}.json"
    )

    print(
        v,
        tok.get_vocab().get("[UNK]")
    )

5000 0
10000 0
15000 0
20000 0
25000 0
30000 0
35000 0
40000 0
45000 0
50000 0


In [76]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "uni_bn_50000.json"
)

samples = [
    "বাংলাদেশ",
    "মুক্তিযুদ্ধ",
    "বিশ্ববিদ্যালয়",
    "আলম ভাই ভালো মানুষ",
    "খানকি মাগী"
]

for s in samples:

    enc = tok.encode(s)

    print("="*60)
    print("TEXT :", s)
    print("TOKENS:")
    print(enc.tokens)

TEXT : বাংলাদেশ
TOKENS:
['বাংলাদেশ']
TEXT : মুক্তিযুদ্ধ
TOKENS:
['মুক্তিযুদ্ধ']
TEXT : বিশ্ববিদ্যালয়
TOKENS:
['বিশ্ববিদ্যালয়']
TEXT : আলম ভাই ভালো মানুষ
TOKENS:
['আলম', 'ভাই', 'ভালো', 'মানুষ']
TEXT : খানকি মাগী
TOKENS:
['খানকি', 'মাগী']


In [77]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(tokenizer_path, test_file):

    tok = Tokenizer.from_file(tokenizer_path)

    total_words = 0
    total_tokens = 0
    oov = 0

    token_lengths = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            words = line.strip().split()

            for w in words:

                total_words += 1

                try:
                    enc = tok.encode(w)
                    tks = enc.tokens

                except:
                    oov += 1
                    continue

                total_tokens += len(tks)

                for t in tks:
                    token_lengths.append(len(t))

    fertility = total_tokens / total_words

    cpt = (
        np.mean(token_lengths)
        if token_lengths else 0
    )

    compression = cpt

    avg_tokens = fertility

    wfr = (
        (total_tokens - total_words)
        / total_tokens
        if total_tokens else 0
    )

    variance = (
        np.var(token_lengths)
        if token_lengths else 0
    )

    return {
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [78]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"uni_bn_{vocab_size}.json",
        "test_litsocial_bn.txt"
    )

    r["vocab"] = vocab_size

    print(r)

    results.append(r)

print("\nFinished.")

{'oov': 0.0, 'fertility': 1.703787669567153, 'cpt': np.float64(2.9206786599055508), 'compression': np.float64(2.9206786599055508), 'avg_tokens': 1.703787669567153, 'wfr': 0.4130724045831075, 'variance': np.float64(3.0055795928387097), 'vocab': 5000}
{'oov': 0.0, 'fertility': 1.5540615738469845, 'cpt': np.float64(3.202071508142804), 'compression': np.float64(3.202071508142804), 'avg_tokens': 1.5540615738469845, 'wfr': 0.35652485279295526, 'variance': np.float64(3.438300838555356), 'vocab': 10000}
{'oov': 0.0, 'fertility': 1.4940829256027337, 'cpt': np.float64(3.3306158595631628), 'compression': np.float64(3.3306158595631628), 'avg_tokens': 1.4940829256027337, 'wfr': 0.3306931075485076, 'variance': np.float64(3.6448183337655626), 'vocab': 15000}
{'oov': 0.0, 'fertility': 1.461144338693256, 'cpt': np.float64(3.405697955867501), 'compression': np.float64(3.405697955867501), 'avg_tokens': 1.461144338693256, 'wfr': 0.31560491765356385, 'variance': np.float64(3.77125568094138), 'vocab': 20000

In [79]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

for vocab_size in VOCABS:

    tokenizer = Tokenizer(Unigram())

    tokenizer.pre_tokenizer = Whitespace()

    trainer = UnigramTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"],
        unk_token="[UNK]"
    )

    tokenizer.train(
        ["train_litsocial_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"uni_iso_{vocab_size}.json"
    )

    print("Saved:", vocab_size)

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [80]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"uni_iso_{vocab_size}.json",
        "test_litsocial_iso.txt"
    )

    r["vocab"] = vocab_size

    print(r)

    results.append(r)

print("\nFinished.")

{'oov': 0.0, 'fertility': 1.7316394544083034, 'cpt': np.float64(3.4349716968107136), 'compression': np.float64(3.4349716968107136), 'avg_tokens': 1.7316394544083034, 'wfr': 0.42251258051769364, 'variance': np.float64(4.223452505423727), 'vocab': 5000}
{'oov': 0.0, 'fertility': 1.5887493353228992, 'cpt': np.float64(3.7439087354610057), 'compression': np.float64(3.7439087354610057), 'avg_tokens': 1.5887493353228992, 'wfr': 0.37057408757514354, 'variance': np.float64(4.752614649478058), 'vocab': 10000}
{'oov': 0.0, 'fertility': 1.531494359003953, 'cpt': np.float64(3.883874909498061), 'compression': np.float64(3.883874909498061), 'avg_tokens': 1.531494359003953, 'wfr': 0.3470429753000358, 'variance': np.float64(4.984770866504712), 'vocab': 15000}
{'oov': 0.0, 'fertility': 1.5015684318566853, 'cpt': np.float64(3.9612796784881916), 'compression': np.float64(3.9612796784881916), 'avg_tokens': 1.5015684318566853, 'wfr': 0.3340296860373505, 'variance': np.float64(5.123498334696965), 'vocab': 20

# Literature + Social Media Corpus Tokenization Evaluation

## Overview

This notebook evaluates the impact of corpus composition, script representation, and tokenizer choice on Bangla subword tokenization. A combined corpus was created by merging Bengali literature and social media text to investigate how tokenizers behave when exposed to both formal and highly informal language.

The study compares three widely used subword tokenization methods:

* Byte Pair Encoding (BPE)
* WordPiece
* Unigram

Experiments are conducted on both:

* Native Bangla script
* ISO-15919 transliterated script

Multiple vocabulary sizes ranging from 5,000 to 50,000 tokens are evaluated using intrinsic tokenization metrics.

---

## Dataset Construction

The corpus was created by combining:

* 250,000 literature documents
* 62,617 social media documents

The resulting dataset contains approximately:

* Documents: 125,234
* Characters: 14.52 million
* Words: 2.43 million
* Vocabulary: 274,747 unique words
* Type-Token Ratio (TTR): 0.1133
* Average Word Length: 4.98 characters

The dataset was split into:

| Split | Documents |
| ----- | --------: |
| Train |   112,710 |
| Test  |    12,524 |

Corresponding ISO-15919 transliterations were generated for all documents.

---

## Script Analysis

### Native Bangla

* Characters: 22,115,321
* Words: 3,500,357
* Vocabulary: 321,674
* Average Word Length: 6.32

### ISO-15919

* Characters: 25,781,441
* Words: 3,500,323
* Vocabulary: 309,388
* Average Word Length: 7.37

### Transliteration Statistics

* Expansion Factor: 1.166
* Vocabulary Ratio: 0.980

The ISO representation increases character length while slightly reducing vocabulary diversity.

---

## Transliteration Validation

A manual sanity check was performed on randomly selected examples.

* Exact Matches: 51 / 100
* Accuracy: 0.51

Most differences were orthographic normalizations rather than semantic changes.

---

## Tokenization Experiments

For each tokenizer:

* Vocabulary sizes: 5k–50k
* Scripts: Bangla and ISO
* Evaluation corpus: Literature + Social Media test set

Metrics reported:

* OOV Rate
* Fertility
* Characters per Token (CPT)
* Compression Ratio
* Average Tokens per Word
* Word Fragmentation Rate (WFR)
* Segmentation Variance

---

## Key Findings

### Effect of Vocabulary Size

Across all tokenizers, increasing vocabulary size:

* Reduced fertility
* Reduced fragmentation
* Increased compression efficiency

Performance gains became progressively smaller beyond approximately 30k–35k vocabulary size, indicating diminishing returns from further vocabulary expansion.

### Comparison of Tokenizers

BPE consistently achieved:

* Lowest fertility
* Lowest fragmentation rate
* Highest compression efficiency

WordPiece produced results very close to BPE but with slightly higher fragmentation.

Unigram achieved zero OOV rate but generated the most fragmented segmentations and lower compression efficiency.

### Effect of Script Representation

ISO-15919 consistently produced:

* Higher CPT values
* Higher compression ratios
* Similar OOV rates

The transliterated representation therefore allows larger average token lengths while preserving lexical coverage.

---

## Overall Conclusion

The Literature + Social Media corpus demonstrates that tokenizer behavior remains consistent across mixed-domain Bangla text. BPE provides the most compact segmentation, WordPiece offers comparable performance, and Unigram favors coverage at the cost of increased fragmentation.

The results further indicate that ISO-15919 transliteration improves compression-related metrics while maintaining comparable lexical coverage, making it a useful alternative representation for Bangla tokenization research.
